# Conversational Memory — Buffer, Window, Summary & Entity

Give your LLM a memory — maintain context across conversation turns with different strategies.

---

In [ ]:
!pip install -q langchain langchain-openai

---

## 1 · The Problem — LLMs Are Stateless

**The Problem** — Every LLM API call is independent. The model has zero memory of previous interactions.

**The Solution** — Memory modules store conversation history and inject it into the prompt, giving the illusion of continuity.

> An LLM without memory is like a goldfish. Memory is the notebook you hand it before every conversation so it can read what happened before.

In [ ]:
from langchain_openai import ChatOpenAI

# ── Without memory — the LLM forgets everything ──
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Two separate calls — the model has NO idea about the first one
response1 = llm.invoke("Hi! My name is Hitesh and I work on LangChain tutorials.")
print(f"Response 1: {response1.content}\n")

response2 = llm.invoke("What's my name and what do I work on?")
print(f"Response 2: {response2.content}")
print("\n→ The model has no idea who you are. Each call is independent.")

### What Just Happened?

The second call has no context from the first. The model can't answer "What's my name?" because it never received that information in the second request. Memory solves this.

---

## 2 · ConversationBufferMemory — Full History

**The Problem** — You need complete conversation fidelity. Every word matters.

**The Solution** — Store the entire conversation verbatim and replay it on every turn.

> A court stenographer that transcribes every word. Complete record, but the transcript keeps growing.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_openai import ChatOpenAI

# ── Set up a conversation with buffer memory ──
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationBufferMemory(return_messages=True)  # store as Message objects

chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True    # shows the full prompt sent to the LLM (educational)
)

# Turn 1
print(chain.predict(input="Hi! My name is Hitesh and I'm building LangChain tutorials."))
print()

In [ ]:
# Turn 2 — the model now remembers!
print(chain.predict(input="What's my name and what am I working on?"))
print()

# Turn 3
print(chain.predict(input="I'm also interested in RAG pipelines."))

In [ ]:
# ── Inspect what's stored in memory ──
print("=== Memory Contents ===")
for msg in memory.chat_memory.messages:
    role = "Human" if msg.type == "human" else "AI"
    print(f"  {role}: {msg.content[:100]}..." if len(msg.content) > 100 else f"  {role}: {msg.content}")

print(f"\nTotal messages stored: {len(memory.chat_memory.messages)}")
print("\n→ Every message is stored verbatim. Token count grows linearly.")

### What Just Happened?

- `ConversationBufferMemory` stored every Human/AI exchange
- On each turn, the full history was injected into the prompt
- The model correctly recalled your name and interests across turns
- **Tradeoff:** Token count grows linearly. A 50-turn conversation can exceed context limits

> **When to use:** Short conversations (< 10-15 exchanges) where you need every detail.

---

## 3 · ConversationBufferWindowMemory — Sliding Window

**The Problem** — Full history grows forever and wastes tokens on old exchanges.

**The Solution** — Keep only the last `k` exchanges. Older messages are dropped.

> A whiteboard that only fits 5 items. When you add a 6th, you erase the oldest.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# ── Sliding window with k=2 (only remember last 2 exchanges) ──
memory_window = ConversationBufferWindowMemory(k=2, return_messages=True)
chain_window = ConversationChain(llm=llm, memory=memory_window, verbose=False)

# Turn 1
chain_window.predict(input="My name is Hitesh.")

# Turn 2
chain_window.predict(input="I live in Charlotte, NC.")

# Turn 3
chain_window.predict(input="My favorite language is Python.")

# Turn 4 — can the model still remember Turn 1?
response = chain_window.predict(input="What's my name?")
print(f"Q: What's my name?")
print(f"A: {response}\n")

# ── Check what's actually in the window ──
print("=== Window Contents (k=2) ===")
for msg in memory_window.chat_memory.messages:
    role = "Human" if msg.type == "human" else "AI"
    print(f"  {role}: {msg.content[:80]}")

print(f"\n→ Only the last {2} exchanges are kept. Turn 1 (name) was dropped.")

### What Just Happened?

- With `k=2`, only the last 2 human/AI exchanges are retained
- Turn 1 (your name) was dropped from the window, so the model likely couldn't answer
- **Tradeoff:** Fixed token cost, but no memory of anything before the window

> **When to use:** When only recent context matters and you want predictable token usage.

---

## 4 · ConversationSummaryMemory — Compressed History

**The Problem** — You need long-term context but can't afford unlimited tokens.

**The Solution** — An LLM progressively summarizes the conversation. The summary replaces full history.

> Instead of keeping every page of meeting notes, you maintain a one-paragraph executive summary that gets updated after each meeting.

In [ ]:
from langchain.memory import ConversationSummaryMemory

# ── Summary memory — uses an LLM to compress history ──
memory_summary = ConversationSummaryMemory(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),  # LLM for summarization
    return_messages=True
)

chain_summary = ConversationChain(llm=llm, memory=memory_summary, verbose=False)

# Have a multi-turn conversation
chain_summary.predict(input="Hi, I'm Hitesh. I'm an AI engineer.")
chain_summary.predict(input="I'm building a LangChain tutorial series.")
chain_summary.predict(input="So far I've covered basics, LCEL, parsers, loaders, splitters, and RAG.")
chain_summary.predict(input="The next tutorial is about conversational memory.")

# ── View the summary ──
print("=== Conversation Summary ===")
print(memory_summary.buffer)
print(f"\n→ The entire 4-turn conversation is compressed into a short summary.")

In [ ]:
# ── Test recall — does the summary retain key facts? ──
response = chain_summary.predict(input="What tutorials have I completed so far?")
print(f"Q: What tutorials have I completed so far?")
print(f"A: {response}")
print(f"\n→ The summary preserved key facts despite compression.")

### What Just Happened?

- `ConversationSummaryMemory` used a secondary LLM call to compress the conversation into a summary
- The summary is updated progressively after each turn
- Key facts (name, role, tutorials completed) were preserved despite compression
- **Tradeoff:** Slower (extra LLM call per turn), and fine details can be lost. But token growth is logarithmic.

---

## 5 · ConversationSummaryBufferMemory — Best of Both

**The Problem** — Summary memory loses recent details. Buffer memory grows too fast.

**The Solution** — Keep recent messages in full, summarize everything older. Best balance for production.

> Recent messages stay sharp, older context gets compressed. Like a news anchor who reads today's headlines in full but summarizes last week.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory

# ── Summary + buffer hybrid ──
memory_hybrid = ConversationSummaryBufferMemory(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    max_token_limit=100,         # once buffer exceeds this, older messages get summarized
    return_messages=True
)

chain_hybrid = ConversationChain(llm=llm, memory=memory_hybrid, verbose=False)

# Build up conversation history
chain_hybrid.predict(input="Hi, I'm Hitesh. I'm building LangChain tutorials.")
chain_hybrid.predict(input="I've finished tutorials on basics, LCEL, and RAG.")
chain_hybrid.predict(input="The current tutorial is about memory types.")
chain_hybrid.predict(input="After this, I'll cover agents and callbacks.")

# ── Inspect the hybrid memory ──
print("=== Summary (older messages) ===")
print(memory_hybrid.moving_summary_buffer or "(no summary yet — all messages fit in buffer)")
print(f"\n=== Buffer (recent messages) ===")
for msg in memory_hybrid.chat_memory.messages[-4:]:
    role = "Human" if msg.type == "human" else "AI"
    print(f"  {role}: {msg.content[:80]}")

In [ ]:
# ── Test recall of both old and recent info ──
print("Q: What's my name?")
print(f"A: {chain_hybrid.predict(input='What is my name?')}\n")

print("Q: What will I cover next?")
print(f"A: {chain_hybrid.predict(input='What topics will I cover next?')}")

### What Just Happened?

- Older messages were summarized when the buffer exceeded `max_token_limit`
- Recent messages stayed in the buffer with full fidelity
- The model could answer questions about both old (name) and recent (next topics) information
- **This is the most practical choice for production chatbots**

---

## 6 · ConversationEntityMemory — Tracking Facts

**The Problem** — In long conversations, the model forgets specific facts about people, places, and things.

**The Solution** — Maintain a structured store of entities mentioned in conversation, updated after each turn.

> A CRM that tracks every person and company mentioned, updating profiles as new facts emerge.

In [ ]:
from langchain.memory import ConversationEntityMemory
from langchain.chains import ConversationChain

# ── Entity memory — tracks structured info about entities ──
memory_entity = ConversationEntityMemory(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    return_messages=True
)

chain_entity = ConversationChain(llm=llm, memory=memory_entity, verbose=False)

# Mention several entities
chain_entity.predict(input="Hitesh is an AI engineer who builds LangChain tutorials.")
chain_entity.predict(input="Alice is Hitesh's colleague. She specializes in MLOps.")
chain_entity.predict(input="They both work at a startup called OrinLabs.")

# ── View the entity store ──
print("=== Entity Store ===")
for entity, description in memory_entity.entity_store.store.items():
    print(f"  {entity}: {description}")

In [ ]:
# ── Query about entities ──
print("Q: What does Alice do?")
print(f"A: {chain_entity.predict(input='What does Alice specialize in?')}\n")

print("Q: Where do Hitesh and Alice work?")
print(f"A: {chain_entity.predict(input='Where do Hitesh and Alice work?')}")

### What Just Happened?

- `ConversationEntityMemory` extracted entities (Hitesh, Alice, OrinLabs) and stored structured profiles
- Each entity's profile is updated as new facts are mentioned
- The model can answer questions about specific entities using these profiles

> **When to use:** Support bots tracking customer details, personal assistants remembering preferences, or any scenario where structured entity recall matters.

---

## 7 · Side-by-Side Comparison

Same conversation, different memory types.

In [ ]:
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationSummaryBufferMemory,
)

# ── Configure all memory types ──
memories = {
    "Buffer": ConversationBufferMemory(return_messages=True),
    "Window (k=2)": ConversationBufferWindowMemory(k=2, return_messages=True),
    "Summary": ConversationSummaryMemory(
        llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
        return_messages=True
    ),
    "SummaryBuffer": ConversationSummaryBufferMemory(
        llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
        max_token_limit=100,
        return_messages=True
    ),
}

# ── Same conversation through each memory type ──
messages = [
    "Hi, I'm Hitesh.",
    "I'm building LangChain tutorials.",
    "I've covered RAG and vector stores.",
    "Now I'm learning about memory.",
]

for name, mem in memories.items():
    chain = ConversationChain(llm=llm, memory=mem, verbose=False)
    for msg in messages:
        chain.predict(input=msg)
    answer = chain.predict(input="What's my name and what have I covered?")
    print(f"{'='*60}")
    print(f"{name}:")
    print(f"  {answer}")
    print()

---

## Key Takeaways

| Memory Type | Stores | Token Growth | Best For |
|-------------|--------|-------------|----------|
| Buffer | Full conversation | Linear | Short conversations |
| BufferWindow | Last K exchanges | Fixed | Recent context only |
| Summary | Compressed summary | Logarithmic | Long conversations |
| SummaryBuffer | Summary + recent buffer | Bounded | Production chatbots |
| Entity | Entity profiles | Per-entity | Tracking people/things |

**Practical advice:** Start with `ConversationBufferMemory` for prototyping. Move to `ConversationSummaryBufferMemory` for production. Add `ConversationEntityMemory` if you need structured fact tracking.

**Next →** [09 · Agents & Tools](../09-agents-tools/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*